# Pipeline complet : reconstruction → soumission

1. **Expositions** : une ridge par allocation, `target(d) ~ actions de sa bourse à la séance suivante`, sur le train daté
2. **Datation** des dates isolées : on cherche le jour où les vrais rendements expliquent le mieux l'historique de 20 jours (tous les groupes du `TS` votent ensemble)
3. **Mise à jour** des expositions avec les historiques des dates trouvées (le test est des années après le train), puis on re-date
4. **Prédiction** : signe de `expositions · rendements des actions à la séance suivante`

D'abord une **simulation** sur le train (on cache des dates 2013-2015 et on fait comme si c'était le test), puis le **vrai run** sur le test.

Données : `calendar/train_dates.csv` (`calendar_dating.ipynb`), `market_data/*_returns.csv` (`download_stocks.py`). Un groupe sans fichier d'actions utilise seulement son indice.

⚠️ Ce n'est pas une prévision : les rendements réels du jour de la target sont utilisés. Vérifier le règlement du challenge (données externes).

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from utils import reconstruction as rc

X = pd.read_csv("data/X_train.csv", index_col="ROW_ID")
y = pd.read_csv("data/y_train.csv", index_col="ROW_ID")["target"]
X_test = pd.read_csv("data/X_Test.csv", index_col="ROW_ID")
dates = pd.read_csv("calendar/train_dates.csv", parse_dates=["date"])
index_returns = pd.read_csv("market_data/returns.csv", index_col="Date", parse_dates=True)

X["target"] = y
reliable = dates[dates["reliable"] & dates["date"].notna()]
X["date"] = X["TS"].map(dict(zip(reliable["TS"], reliable["date"])))
train = X.dropna(subset=["date"]).copy()
print(len(train), "dated train rows,", train["date"].min().date(), "->", train["date"].max().date())

514530 dated train rows, 2006-06-09 -> 2015-10-30


In [2]:
markets = {}
for g in [1, 2, 3, 4]:
    first_day = train.loc[train["GROUP"] == g, "date"].min()
    markets[g] = rc.Market(g, index_returns, first_day, std_start=first_day, std_end="2015-11-01")
    rows = train["GROUP"] == g
    train.loc[rows, "next_date"] = markets[g].next_session(train.loc[rows, "date"])

group 1 (XTKS): 453 stocks + index


group 2 (XHKG): 130 stocks + index


group 3: no stock file (market_data/korea_returns.csv), index only
group 3 (XKRX): 0 stocks + index


group 4: no stock file (market_data/china_returns.csv), index only
group 4 (XSHG): 0 stocks + index


## Penalty of the ridge (one per group)

Fit before 2012-07, checked on 2012-07 → 2013-06.

In [3]:
lams = {}
for g in markets:
    rows = train[train["GROUP"] == g]
    lams[g] = rc.choose_lambda(rows, markets[g], "2012-07-01", "2013-07-01")
lams

{1: 1000, 2: 1000, 3: 100, 4: 100}

## Simulation sur le train

- expositions apprises avant 2013-06
- « faux test » : des dates entre 2013-07 et 2015-10, traitées comme isolées
- dates possibles : tout sauf autour des dates d'apprentissage (comme pour le vrai test)

Deux versions : **dense** (150 dates au hasard, beaucoup de données pour la mise à jour, optimiste) et **réaliste** (une date tous les ~30 jours, sans chevauchement, comme le vrai test, mais seulement ~28 dates donc bruité).

In [4]:
fit_rows = train[train["date"] < "2013-06-01"]
fake_pool = train[train["date"] >= "2013-07-01"]
SPACED = False   # False = dense version, True = realistic version

if SPACED:
    first = fake_pool.drop_duplicates("TS").sort_values("date")
    fake_ts, last = [], None
    for ts, d in zip(first["TS"], first["date"]):
        if last is None or (d - last).days > 30:
            fake_ts.append(ts)
            last = d
else:
    fake_ts = list(fake_pool["TS"].drop_duplicates().sample(150, random_state=0))
fake = fake_pool[fake_pool["TS"].isin(fake_ts)].copy()
true_date = fake.drop_duplicates("TS").set_index("TS")["date"]
print(len(fake_ts), "fake test dates")

betas0, noise = {}, {}
for g in markets:
    betas0[g], noise[g] = rc.fit_exposures(fit_rows[fit_rows["GROUP"] == g], markets[g], lams[g])
    print(f"group {g}: exposures for {len(betas0[g])} allocations")

LAST_DAY = index_returns.index.max() - pd.Timedelta(days=3)   # we need the next session's returns
allowed = rc.allowed_days(markets, fit_rows["date"].unique(), end=LAST_DAY)
print(len(allowed), "possible days")

150 fake test dates


group 1: exposures for 70 allocations


group 2: exposures for 70 allocations


group 3: exposures for 70 allocations


group 4: exposures for 66 allocations
4070 possible days


In [5]:
def evaluate(dated, betas, label):
    dated = dated.copy()
    dated["right"] = dated["date"].values == true_date.reindex(dated["TS"]).values
    pred = rc.predict(fake, dated, markets, betas)
    ok = pred.notna()
    acc = ((pred[ok] > 0) == (fake.loc[ok, "target"] > 0)).mean()
    print(f"{label}: dates right {dated['right'].mean():.0%} | accuracy {acc:.1%} on {ok.mean():.0%} of the rows")
    return dated, pred


# with the true dates (what perfect dating would give)
oracle = pd.DataFrame({"TS": true_date.index, "date": true_date.values, "margin": 1.0})
_ = evaluate(oracle, betas0, "true dates, train exposures")
_ = evaluate(oracle, rc.refresh_exposures(fake, oracle, markets, betas0, lams), "true dates, refreshed exposures")

# the real thing: date, refresh the exposures, re-date (no overlap constraint in this simulation)
dated, betas, history = rc.date_with_refresh(fake, markets, betas0, noise, lams, allowed,
                                             min_gap_days=25 if SPACED else 0)
_ = evaluate(dated, betas0, "estimated dates, train exposures")
dated, pred = evaluate(dated, betas, "estimated dates, refreshed exposures")

true dates, train exposures: dates right 100% | accuracy 56.7% on 100% of the rows


true dates, refreshed exposures: dates right 100% | accuracy 62.5% on 100% of the rows


round 1: median margin 162.5


round 2: median margin 404.9


estimated dates, train exposures: dates right 86% | accuracy 56.1% on 100% of the rows


estimated dates, refreshed exposures: dates right 86% | accuracy 61.4% on 100% of the rows


In [6]:
# is it more often right when the date is clearly the best one ?
fake["margin"] = fake["TS"].map(dict(zip(dated["TS"], dated["margin"])))
fake["right_date"] = fake["TS"].map(dict(zip(dated["TS"], dated["right"])))
fake["correct"] = (pred > 0) == (fake["target"] > 0)
fake["margin_bin"] = pd.qcut(fake["margin"], 4, labels=["low", "mid-low", "mid-high", "high"], duplicates="drop")
fake.groupby("margin_bin")[["right_date", "correct"]].mean().round(3)

,right_date,correct
margin_bin,,
low,0.517,0.552
mid-low,0.943,0.594
mid-high,1.000,0.654
high,1.000,0.657


In [7]:
fake.groupby("GROUP")[["right_date", "correct"]].mean().round(3)

,right_date,correct
GROUP,,
1,0.858,0.712
2,0.858,0.666
3,0.868,0.530
4,0.869,0.538


## Vrai run sur le test

- expositions apprises sur les 2 dernières années du train
- dates possibles : loin (> 25 jours) de toutes les dates du train
- datation avec les expositions du train, mise à jour des expositions avec les historiques des dates trouvées (validation croisée en deux moitiés), re-datation, puis mise à jour finale pour prédire
- en simulation avec 2,5 ans d'écart entre train et faux test : 71 % de dates justes au 1er tour, 83 % au 2e

In [8]:
recent = train[train["date"] >= train["date"].max() - pd.Timedelta(days=730)]
betas0, noise = {}, {}
for g in markets:
    betas0[g], noise[g] = rc.fit_exposures(recent[recent["GROUP"] == g], markets[g], lams[g], min_rows=60)
    print(f"group {g}: exposures for {len(betas0[g])} allocations")

# the test windows never overlap the train ones -> far from every train date.
# first pass on the whole history to see where the test dates fall
all_train_dates = dates["date"].dropna().unique()
allowed_all = rc.allowed_days(markets, all_train_dates, end=LAST_DAY)
first = rc.date_all(X_test, rc.Dater(markets, betas0, noise), allowed_all, min_gap_days=25)
print(f"without constraint: {(first['date'] >= '2015-11-01').mean():.0%} of the test dates after the end of the train")
print(first.groupby(first["date"] >= "2015-11-01")["margin"].median().rename("median margin"))

group 1: exposures for 70 allocations
group 2: exposures for 70 allocations


group 3: exposures for 70 allocations
group 4: exposures for 66 allocations


without constraint: 82% of the test dates after the end of the train
date
False   -30.542270
True     38.463117
Name: median margin, dtype: float64


In [9]:
# the test is after the train: we only look there
TEST_START = "2015-11-01"
allowed = rc.allowed_days(markets, all_train_dates, start=TEST_START, end=LAST_DAY)
print(len(allowed), "possible days for the test")

test_dated, betas, history = rc.date_with_refresh(X_test, markets, betas0, noise, lams, allowed, min_gap_days=25)
print(test_dated["date"].dt.year.value_counts().sort_index())

2547 possible days for the test


round 1: median margin 7.1


round 2: median margin 112.3


date
2015     1
2016    11
2017    12
2018    13
2019    14
2020    12
2021     8
2022    12
2023    10
2024    11
2025    12
2026     4
Name: count, dtype: int64


In [10]:
pred_test = rc.predict(X_test, test_dated, markets, betas)
print(f"prediction for {pred_test.notna().mean():.1%} of the test rows, the rest -> 1")

submission = pd.read_csv("data/sample_submission.csv", index_col="ROW_ID")
submission["prediction"] = (pred_test.reindex(submission.index) > 0).astype(int)
submission.loc[pred_test.reindex(submission.index).isna(), "prediction"] = 1
submission.to_csv("data/submission_reconstruction.csv")
test_dated.to_csv("calendar/test_dates_reconstruction.csv", index=False)

print(submission["prediction"].value_counts(normalize=True).round(3))

prediction for 99.4% of the test rows, the rest -> 1
prediction
0    0.524
1    0.476
Name: proportion, dtype: float64
